# Seed Ablation

Runs multiple random seeds for selected architectures. The base config exposes optional force loss and Hessian regularization.

In [1]:
import os
import jax.numpy as jnp

from properties import SlinkyN3Properties
from run_architectures import SweepConfig, subset_energy_only, subset_main_paper_candidates, subset_all
from seed_ablation_utils import (
    run_seed_ablation,
    print_seed_summary,
    save_seed_summary_json,
    make_seed_ablation_plots,
)


In [2]:
# Dataset / physical setup
properties = SlinkyN3Properties(mass=0.3)
train_file = "../simulation_data_2D/3_noded/n3_slinky_sim_train_dataset.npz"
valid_file = "../simulation_data_2D/3_noded/n3_slinky_sim_test_dataset.npz"

# selected_architectures = subset_energy_only()
# selected_architectures = subset_main_paper_candidates()
# selected_architectures = subset_all()

# remove mlp_energy, chol_stiffness_signed_mlp, all stiffness_icnn from the plot since they are so bad that they throw off the scale and make it hard to see differences between the other architectures
selected_architectures = [
    arch for arch in subset_all()
    if arch not in ["chol_stiffness_signed_mlp", "diag_stiffness_icnn", "chol_stiffness_icnn", "chol_stiffness_signed_icnn"]
]

# Optional force loss. Leave strength at 0.0 for displacement-only training.
force_loss_strength = 1.0
force_key = None
force_components = (0,)
force_sign = 1.0
return_loss_components = force_loss_strength != 0.0

# Optional Hessian regularizer.
hessian_reg_strength = 0.0
hessian_reg_probes = 1
hessian_reg_seed = 0


In [3]:
base_cfg = SweepConfig(
    output_dir="seed_ablation_outputs_n3_slinky_simdata_force_optional",
    n_epochs=500,
    lr=1e-2,
    seed=0,
    seed_list=tuple(range(25)),
    hidden=(10,),
    input_mode="invariant",
    activation="tanh",
    corr_factor=0.01,
    only_stretching_NN=False,
    zero_reference=True,
    valid_every=1,
    max_dlambda=5e-2,
    iters=20,
    ls_steps=10,
    abs_tol=1e-4,
    rel_tol=1e-4,
    train_fail_on_nonconvergence=False,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=hessian_reg_strength,
    hessian_reg_probes=hessian_reg_probes,
    hessian_reg_seed=hessian_reg_seed,
    force_key=force_key,
    force_loss_strength=force_loss_strength,
    force_components=force_components,
    force_sign=force_sign,
    return_loss_components=return_loss_components,
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_energy_landscapes=True,
    verbose=True,
)


In [4]:
all_seed_results = run_seed_ablation(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    base_cfg=base_cfg,
    selected_architectures=selected_architectures,
)

print_seed_summary(all_seed_results)



SEED ABLATION for architecture: diag_energy_baseline

--- Running seed 0 for diag_energy_baseline ---

Running architecture: diag_energy_baseline
  model_cls               : DiagonalPlusEnergyNN
  which_case              : baseline
  hidden                  : (10,)
  input_mode              : invariant
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.01
  zero_reference          : True
  seed                    : 0
  max_dlambda             : 0.05
  iters                   : 20
  ls_steps                : 10
  abs_tol                 : 0.0001
  rel_tol                 : 0.0001
  training fail_on_nonconvergence       : False
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  hessian_reg_strength    : 0.0
  hessian_reg_probes      : 1
  hessian_reg_seed        : 0
  force_key               : None
  force_loss_strength     : 1.0
  force_components        : (0,)

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/architecture_plots.py:234: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



--- Running seed 1 for diag_energy_baseline ---

Running architecture: diag_energy_baseline
  model_cls               : DiagonalPlusEnergyNN
  which_case              : baseline
  hidden                  : (10,)
  input_mode              : invariant
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.01
  zero_reference          : True
  seed                    : 1
  max_dlambda             : 0.05
  iters                   : 20
  ls_steps                : 10
  abs_tol                 : 0.0001
  rel_tol                 : 0.0001
  training fail_on_nonconvergence       : False
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  hessian_reg_strength    : 0.0
  hessian_reg_probes      : 1
  hessian_reg_seed        : 0
  force_key               : None
  force_loss_strength     : 1.0
  force_components        : (0,)
  force_sign              : 1.0
  exp_dir            

KeyboardInterrupt: 

In [ ]:
summary_dir = os.path.join(base_cfg.output_dir, "seed_ablation_summary")
save_seed_summary_json(all_seed_results, output_dir=summary_dir)
make_seed_ablation_plots(all_seed_results, output_dir=summary_dir, traj_idx=0)
summary_dir
